In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# Load dataset
df = pd.read_csv('marketing_sales_data.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()


Dataset loaded successfully!
Shape: (572, 5)
       TV      Radio  Social Media Influencer       Sales
0     Low   3.518070      2.293790      Micro   55.261284
1     Low   7.756876      2.572287       Mega   67.574904
2    High  20.348988      1.227180      Micro  272.250108
3  Medium  20.108487      2.728374       Mega  195.102176
4    High  31.653200      7.776978       Nano  273.960377


In [2]:
# Check shape, missing values, data types
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Descriptive Statistics ===")
print(df.describe())

print("\n=== Categorical Column Values ===")
print("TV unique values:", list(df['TV'].unique()))
print("Influencer unique values:", list(df['Influencer'].unique()))


=== Data Types ===
TV                  str
Radio           float64
Social Media    float64
Influencer          str
Sales           float64
dtype: object

=== Missing Values ===
TV              0
Radio           0
Social Media    0
Influencer      0
Sales           0
dtype: int64

=== Descriptive Statistics ===
            Radio  Social Media       Sales
count  572.000000    572.000000  572.000000
mean    17.520616      3.333803  189.296908
std      9.290933      2.238378   89.871581
min      0.109106      0.000031   33.509810
25%     10.699556      1.585549  118.718722
50%     17.149517      3.150111  184.005362
75%     24.606396      4.730408  264.500118
max     42.271579     11.403625  357.788195

=== Categorical Column Values ===
TV unique values: ['Low', 'High', 'Medium']
Influencer unique values: ['Micro', 'Mega', 'Nano', 'Macro']


In [3]:
# Encode TV and Influencer as dummy variables (drop_first avoids multicollinearity trap)
df_encoded = pd.get_dummies(df, columns=['TV', 'Influencer'], drop_first=True)

# Rename Social Media column to remove space
df_encoded = df_encoded.rename(columns={'Social Media': 'Social_Media'})

print("=== Encoded Columns ===")
print(df_encoded.columns.tolist())

print("\n=== First 5 Rows ===")
print(df_encoded.head())

print(f"\nShape: {df_encoded.shape}")


=== Encoded Columns ===
['Radio', 'Social_Media', 'Sales', 'TV_Low', 'TV_Medium', 'Influencer_Mega', 'Influencer_Micro', 'Influencer_Nano']

=== First 5 Rows ===
       Radio  Social_Media       Sales  TV_Low  TV_Medium  Influencer_Mega  Influencer_Micro  Influencer_Nano
0   3.518070      2.293790   55.261284    True      False            False              True            False
1   7.756876      2.572287   67.574904    True      False             True             False            False
2  20.348988      1.227180  272.250108   False      False            False              True            False
3  20.108487      2.728374  195.102176   False       True             True             False            False
4  31.653200      7.776978  273.960377   False      False            False             False             True

Shape: (572, 8)


In [4]:
# Convert booleans to integers for correlation
df_encoded = df_encoded.astype({
    'TV_Low': int, 'TV_Medium': int,
    'Influencer_Mega': int, 'Influencer_Micro': int, 'Influencer_Nano': int
})

# Correlation matrix
corr_matrix = df_encoded.corr()

print("=== Correlation with Sales ===")
print(corr_matrix['Sales'].drop('Sales').sort_values(ascending=False))

# Heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Matrix - All Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.close()
print('Heatmap saved to correlation_heatmap.png')


=== Correlation with Sales ===
Radio               0.858036
Social_Media        0.542048
TV_Medium           0.050449
Influencer_Mega     0.032443
Influencer_Nano     0.017656
Influencer_Micro   -0.006503
TV_Low             -0.805896

Heatmap saved to correlation_heatmap.png


In [5]:
# Define predictor variables
X_cols = ['Radio', 'Social_Media', 'TV_Low', 'TV_Medium',
          'Influencer_Mega', 'Influencer_Micro', 'Influencer_Nano']

X_vif = df_encoded[X_cols]

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['Feature'] = X_cols
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i)
                   for i in range(X_vif.shape[1])]

print("=== Variance Inflation Factor (VIF) ===")
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))
print("\nRule: VIF > 10 = severe multicollinearity")
print("      VIF > 5  = moderate multicollinearity")


=== Variance Inflation Factor (VIF) ===
         Feature      VIF
           Radio 6.695980
    Social_Media 5.344600
 Influencer_Nano 2.084801
Influencer_Micro 2.083851
 Influencer_Mega 1.996014
       TV_Medium 1.664554
          TV_Low 1.661129

Rule: VIF > 10 = severe multicollinearity
      VIF > 5  = moderate multicollinearity


## Multicollinearity Check - Findings

### Correlation Matrix Observations
- Radio has the strongest positive correlation with Sales (r = 0.86)
- TV_Low has a strong negative correlation with Sales (r = -0.81)
- Social_Media shows moderate positive correlation (r = 0.54)
- Influencer dummies show weak correlation with Sales (< 0.05)

![Correlation Heatmap](correlation_heatmap.png)

### VIF Results
| Feature         | VIF    | Interpretation         |
|-----------------|--------|------------------------|
| Radio           | 6.70   | Moderate - acceptable  |
| Social_Media    | 5.34   | Moderate - acceptable  |
| Influencer_Nano | 2.08   | Low - no concern       |
| Influencer_Micro| 2.08   | Low - no concern       |
| Influencer_Mega | 2.00   | Low - no concern       |
| TV_Medium       | 1.66   | Low - no concern       |
| TV_Low          | 1.66   | Low - no concern       |

### Conclusion
No severe multicollinearity detected (all VIF < 10). All predictors are retained.


In [7]:
# Define X and y
X = df_encoded[['Radio', 'Social_Media', 'TV_Low', 'TV_Medium',
                'Influencer_Mega', 'Influencer_Micro', 'Influencer_Nano']]
y = df_encoded['Sales']

# Add constant and fit OLS model
X_ols = sm.add_constant(X)
model = sm.OLS(y, X_ols).fit()

# Display summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.904
Model:                            OLS   Adj. R-squared:                  0.903
Method:                 Least Squares   F-statistic:                     760.4
Date:                Sun, 14 Jun 2026   Prob (F-statistic):          1.82e-282
Time:                        23:00:04   Log-Likelihood:                -2713.4
No. Observations:                 572   AIC:                             5443.
Df Residuals:                     564   BIC:                             5478.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const              217.4784      6.584  

## Multiple Linear Regression - Model Results and Interpretation

### Model Equation
Sales = 217.48 + 2.97(Radio) - 0.14(Social_Media) - 154.57(TV_Low)
        - 75.59(TV_Medium) + 2.49(Influencer_Mega) + 2.94(Influencer_Micro)
        + 0.80(Influencer_Nano)

### Model Performance
| Metric            | Value  | Interpretation                          |
|-------------------|--------|-----------------------------------------|
| R-squared         | 0.904  | Model explains 90.4% of Sales variance  |
| Adj. R-squared    | 0.903  | Strong fit after adjusting for predictors|
| F-statistic       | 760.4  | Overall model is highly significant     |
| Prob(F-statistic) | 0.000  | Model is statistically valid            |

### Coefficient Interpretation
**Significant predictors (p < 0.05):**
- **Radio (+2.97):** Each additional $1K in Radio spend increases Sales by $2,974.
- **TV_Low (-154.57):** Low TV spend reduces Sales by $154,574 vs High TV.
- **TV_Medium (-75.59):** Medium TV spend reduces Sales by $75,594 vs High TV.

**Not significant (p > 0.05):**
- **Social_Media (p = 0.837):** No measurable impact on Sales.
- **Influencer categories (p > 0.05):** No significant effect on Sales.


In [9]:
fitted = model.fittedvalues
residuals = model.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.5, edgecolors='k', linewidth=0.3)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Normal Q-Q Plot')

# Scale-Location plot
axes[2].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.5, edgecolors='k', linewidth=0.3)
axes[2].set_xlabel('Fitted Values')
axes[2].set_ylabel('sqrt(|Standardized Residuals|)')
axes[2].set_title('Scale-Location Plot')

plt.suptitle('Multiple Regression - Diagnostic Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('diagnostic_plots.png', dpi=100, bbox_inches='tight')
plt.close()
print('Diagnostic plots saved to diagnostic_plots.png')
print(f'Residual mean: {residuals.mean():.6f}')
print(f'Residual std:  {residuals.std():.4f}')


Diagnostic plots saved to diagnostic_plots.png

Residual mean: -0.000000
Residual std:  27.8181


## Diagnostic Plots - Assumption Validation

![Diagnostic Plots](diagnostic_plots.png)

### 1. Residuals vs Fitted (Linearity and Homoscedasticity)
Residuals are randomly scattered around zero with no clear funnel shape.

### 2. Normal Q-Q Plot (Normality)
Points follow the theoretical line closely in the center with minor tail deviations.

### 3. Scale-Location Plot (Homoscedasticity)
No strong trend in spread across fitted values, supporting constant variance.


## Final Report - Multi-Channel Marketing ROI Recommendation

### Model Summary
A Multiple Linear Regression model was built using TV spend level, Radio budget,
Social Media spend, and Influencer category to predict Sales (R-squared = 0.904).

### Significant Drivers
| Predictor    | Coefficient | p-value | Business Impact                        |
|--------------|-------------|---------|----------------------------------------|
| Radio        | +2.97       | 0.000   | +$2,974 Sales per $1K Radio spend      |
| TV_Low       | -154.57     | 0.000   | -$154,574 vs High TV spend             |
| TV_Medium    | -75.59      | 0.000   | -$75,594 vs High TV spend              |
| Social_Media | -0.14       | 0.837   | Not significant                        |
| Influencer   | varies      | > 0.05  | Not significant                        |

### Recommendations
1. **Maximize TV spend at HIGH level** - largest sales lever ($154,574 gap vs Low TV).
2. **Increase Radio budget** - strongest continuous positive driver ($2,974 per $1K).
3. **Deprioritize Social Media** - not statistically significant (p = 0.837).
4. **Influencer type does not matter** - select based on cost efficiency only.

### Conclusion
Focus budget on High TV spend and Radio investment. Social Media and Influencer
partnerships do not demonstrably drive Sales in this dataset.
